# Reproducing Burman et al. (2023): the network model (Figures 4E to 4H, S7)

**Paper.** Burman RJ, Brodersen PJN, Raimondo JV, Sen A, Akerman CJ (2023). Active cortical networks
promote shunting fast synaptic inhibition in vivo. *Neuron* 111, 3531-3540.
[doi:10.1016/j.neuron.2023.08.005](https://doi.org/10.1016/j.neuron.2023.08.005)

**Data.** These claims are `data: elsewhere`: their data are generated by the paper's simulation
code, gist.github.com/paulbrodersen/2ce61e4e57c7659b727d30e3e1e0128f at revision
f6a05661c10051560aa381e334099efd8521617b, transcribed in `network.py` (its docstring lists the
differences, none of which changes the model). They use nothing from DANDI:000458 (version
0.230317.0039, read in `reproduce_burman_2023.ipynb`; the version constant is `npx.VERSION`).

**Text the claims were written from.** The version of record (PMC11913778) with STAR Methods; the
supplement (Figure S7) was not available.

| Claim | Here |
|---|---|
| fig4f-model-hyperpolarizing-less-activity | tested |
| fig4h-model-synchrony | tested |
| fig4h-model-entropy | tested |
| figs7-model-decoding | tested |

Method (every claim's criterion): the gist at its defaults (100 patterns, 100 repeats per
condition, 800 excitatory and 200 inhibitory neurons, E_GABA -60 mV then -80 mV in the excitatory
neurons), run 20 times (Methods, "Each simulation was repeated 20 times"); per run and condition,
the gist's entropy, synchrony and peri-stimulus rate of each excitatory neuron from spikes 0 to 12 ms
after each input of the first stimulus set, and its decoding accuracy; the 20 runs pooled (16,000
neurons). Seeds are fixed here, 1 to 20, one per run (the gist sets none).

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import time, resource
from multiprocessing import Pool
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import network, npx

pd.set_option("display.width", 200)
T0 = time.time()
SEEDS = list(range(1, 21))
print("dataset named for the record:", npx.DANDISET, npx.VERSION, "(not used by this notebook)")

## Data check

The model's data are its simulated spikes. First a check that the stimulus held as one periodic set (`network.py`) gives the same spikes as the gist's whole-run array, on a small network run (10 patterns, 5 repeats) with one seed.

In [ ]:
a = network.simulate(7, stimuli=10, repeats=5, periodic=True, spikes=True)
b = network.simulate(7, stimuli=10, repeats=5, periodic=False, spikes=True)
(ta, ia), (tb, ib) = a["spikes"], b["spikes"]
print("spikes, periodic stimulus:", len(ta), "| gist's whole-run array:", len(tb))
print("identical spike trains:", len(ta) == len(tb) and np.array_equal(ia, ib) and np.allclose(ta, tb))
print("inhibitory-to-excitatory summed weight (nS): start %.0f, after balancing %.1f, after test %.1f"
      % tuple(a["w_sums"]))

Then a check that `network.py` transcribes the gist: the gist itself, fetched at its revision and
run unmodified with the same small options and seed (`network.run_gist`; its docstring lists what
is changed around it: a stand-in for umap, CSR in place of COO for the installed scipy).

In [ ]:
# The paper's simulation gist, run unmodified (network.run_gist) on the same small network run
# and seed as `a` above, against network.simulate spike for spike, and per neuron for the last
# condition the gist analyses (hyperpolarizing, -80 mV), whose arrays its namespace keeps.
g = network.run_gist(7, stimuli=10, repeats=5)
tg = np.asarray(g["spike_monitor"]["exc"].t / network.b2.ms)
ig = np.asarray(g["spike_monitor"]["exc"].i)
print("excitatory spikes, gist:", len(tg), "| network.simulate:", len(ta))
print("identical spike trains:", len(ta) == len(tg) and np.array_equal(ia, ig) and np.allclose(ta, tg))
for theirs, ours in (("spike_counts", "spike_count"), ("firing_rates", "firing_rate"), ("entropy", "entropy"), ("synchrony", "synchrony")):
    x, y = np.asarray(g[theirs], float), a[-80][ours]
    print(f"{theirs} (-80 mV): equal in {int(np.isclose(x, y, rtol=1e-12, atol=0, equal_nan=True).sum())} of {len(y)} neurons")

In [ ]:
t = time.time()
with Pool(4) as pool:
    runs = pool.map(network.run, SEEDS)
print(f"20 runs in {(time.time() - t) / 60:.1f} min (4 processes); peak memory of this process "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB, of a child "
      f"{resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss / 1e6:.2f} GB")
rows = []
for seed, r in zip(SEEDS, runs):
    for eg, cond in zip(network.EGABA, ("shunting", "hyperpolarizing")):
        d = r[eg]
        rows.append(dict(seed=seed, condition=cond, E_GABA=eg, spikes_in_condition=d["total_spikes"],
                         neurons_with_spikes=int((d["spike_count"] > 0).sum()),
                         rate_mean=d["firing_rate"].mean(), entropy_mean=np.nanmean(d["entropy"]),
                         synchrony_mean=np.nanmean(d["synchrony"]), accuracy=d["accuracy"]))
per_run = pd.DataFrame(rows)
print(per_run.round(4).to_string())
print("\nweights after balancing equal weights after the test in every run:",
      all(abs(r["w_sums"][1] - r["w_sums"][2]) < 1e-6 for r in runs))
print("eta in the plastic synapses' compiled code (pre, post), balancing run / test run:",
      sorted({str(r["eta_used"]) for r in runs}))

The gist run unmodified gives the same 19,093 excitatory spikes as `network.simulate` on the
small run, and the same spike count, rate, entropy and synchrony for all 800 excitatory neurons
of its hyperpolarizing condition. In all 20 full runs every excitatory neuron spikes in both
conditions, the summed inhibitory weight is the same after the test as after balancing, and the
compiled code of the plastic synapses used eta = 0.1 during balancing and eta = 0 during the
test, as the gist's loop sets it.

In [ ]:
def pooled(key):
    sh = np.concatenate([r[-60][key] for r in runs])
    hy = np.concatenate([r[-80][key] for r in runs])
    return sh, hy
def paired_report(key, label):
    sh, hy = pooled(key)
    ok = ~np.isnan(sh) & ~np.isnan(hy)
    t = stats.ttest_rel(sh[ok], hy[ok])
    print(f"{label}: neurons {len(sh)}, defined in both conditions {ok.sum()} (dropped {(~ok).sum()})")
    print(f"  shunting {sh[ok].mean():.4f} +- {stats.sem(sh[ok]):.4f} (SEM), hyperpolarizing {hy[ok].mean():.4f} +- {stats.sem(hy[ok]):.4f}")
    print(f"  paired t test t = {t.statistic:.2f}, p = {t.pvalue:.2e}")
    print(f"  range shunting {np.nanmin(sh):.3f} to {np.nanmax(sh):.3f}, hyperpolarizing {np.nanmin(hy):.3f} to {np.nanmax(hy):.3f}")
    return sh[ok].mean(), hy[ok].mean(), t

## fig4h-model-synchrony

> Model pyramidal neurons are less synchronous with shunting than with hyperpolarizing E_GABAAR
> (0.784 against 0.881).

In [ ]:
syn_sh, syn_hy, syn_t = paired_report("synchrony", "synchrony")
print("expected shunting 0.784 +- 0.001, hyperpolarizing 0.881 +- 0.001")
pr = per_run.pivot(index="seed", columns="condition", values="synchrony_mean")
print("runs with lower mean synchrony under shunting: %d of %d; per-run means shunting %.4f to %.4f, hyperpolarizing %.4f to %.4f"
      % ((pr.shunting < pr.hyperpolarizing).sum(), len(pr), pr.shunting.min(), pr.shunting.max(), pr.hyperpolarizing.min(), pr.hyperpolarizing.max()))
# the spread between seeds: standard error of the 20-run mean (SD of the per-run means / sqrt(20)),
# and the distance of each pooled mean from the paper's value in units of it
for cond, mean, paper in (("shunting", syn_sh, 0.784), ("hyperpolarizing", syn_hy, 0.881)):
    se = pr[cond].std(ddof=1) / np.sqrt(len(pr))
    print(f"{cond}: run-to-run SE of the mean {se:.4f}; pooled mean {mean:.4f} is {abs(mean - paper) / se:.1f} SE from {paper}")

**Result: partially reproduced.** Over the 16,000 pooled neurons synchrony is lower with shunting E_GABA, 0.7833 against 0.8789 (paired t = -134.60; p underflows to 0, below 0.001), and lower in all 20 runs. The shunting mean is within 0.784 +- 0.001; the hyperpolarizing mean is 0.0011 below the lower bound of 0.881 +- 0.001. The per-run hyperpolarizing means span 0.8661 to 0.8909, and the standard error of a 20-run mean from that spread is 0.0013, so the tolerance, the paper's SEM over pooled neurons, is narrower than the spread between runs with other seeds: the hyperpolarizing mean is 1.6 of these SEs from 0.881, the shunting mean 1.3 from 0.784.

## fig4h-model-entropy

> Model pyramidal neurons have higher peri-stimulus entropy with shunting than with
> hyperpolarizing E_GABAAR (0.451 against 0.197 nats).

In [ ]:
ent_sh, ent_hy, ent_t = paired_report("entropy", "entropy (nats)")
print("expected shunting 0.451 +- 0.002, hyperpolarizing 0.197 +- 0.002")
pr = per_run.pivot(index="seed", columns="condition", values="entropy_mean")
print("runs with higher mean entropy under shunting: %d of %d; per-run means shunting %.4f to %.4f, hyperpolarizing %.4f to %.4f"
      % ((pr.shunting > pr.hyperpolarizing).sum(), len(pr), pr.shunting.min(), pr.shunting.max(), pr.hyperpolarizing.min(), pr.hyperpolarizing.max()))
# the spread between seeds: standard error of the 20-run mean (SD of the per-run means / sqrt(20)),
# and the distance of each pooled mean from the paper's value in units of it
for cond, mean, paper in (("shunting", ent_sh, 0.451), ("hyperpolarizing", ent_hy, 0.197)):
    se = pr[cond].std(ddof=1) / np.sqrt(len(pr))
    print(f"{cond}: run-to-run SE of the mean {se:.4f}; pooled mean {mean:.4f} is {abs(mean - paper) / se:.1f} SE from {paper}")
rng = np.random.default_rng(4)  # this section's generator: the 1,000 neurons plotted, as in Figure 4G
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, key, lab in zip(axes, ("synchrony", "entropy"), ("Synchrony", "Entropy (nats)")):
    sh, hy = pooled(key)
    ok = np.flatnonzero(~np.isnan(sh) & ~np.isnan(hy))
    pick = rng.choice(ok, 1000, replace=False)
    ax.scatter(hy[pick], sh[pick], s=3)
    lim = [min(sh[pick].min(), hy[pick].min()), max(sh[pick].max(), hy[pick].max())]
    ax.plot(lim, lim, "k--", lw=1); ax.set_xlabel(lab + ", hyperpolarizing"); ax.set_ylabel(lab + ", shunting")
    print(f"{key}: plotted 1,000 of {len(ok)} neurons, range {lim[0]:.3f} to {lim[1]:.3f}, none outside the axes")
fig.tight_layout(); plt.show()

**Result: partially reproduced.** Entropy is higher with shunting E_GABA, 0.4516 against 0.2014 nats (paired t = 93.09; p underflows to 0, below 0.001), and higher in all 20 runs. The shunting mean is within 0.451 +- 0.002; the hyperpolarizing mean is 0.0024 above the upper bound of 0.197 +- 0.002. Per-run hyperpolarizing means span 0.1798 to 0.2217, wider than the tolerance; the standard error of a 20-run mean from that spread is 0.0023, and the hyperpolarizing mean is 1.9 of these SEs from 0.197 (the shunting mean 0.4 from 0.451).

## fig4f-model-hyperpolarizing-less-activity

> In the network model, shunting E_GABAAR gives more variable input-evoked spike times, while
> hyperpolarizing E_GABAAR gives less activity and spiking more closely aligned across neurons.

The activity part: the gist's peri-stimulus firing rate of the excitatory neurons, pooled over
the 20 runs, neurons with no spike included at zero, paired t test. The spike-time parts are judged
in the two claims above.

In [ ]:
sh, hy = pooled("firing_rate")
t = stats.ttest_rel(sh, hy)
print(f"neurons {len(sh)}; peri-stimulus rate shunting {sh.mean():.3f} +- {stats.sem(sh):.3f} Hz, "
      f"hyperpolarizing {hy.mean():.3f} +- {stats.sem(hy):.3f} Hz; paired t = {t.statistic:.2f}, p = {t.pvalue:.2e}")
print("neurons with no peri-stimulus spike: shunting %d, hyperpolarizing %d" % ((sh == 0).sum(), (hy == 0).sum()))
pr = per_run.pivot(index="seed", columns="condition", values="rate_mean")
print("runs with lower mean rate under hyperpolarizing: %d of %d" % ((pr.hyperpolarizing < pr.shunting).sum(), len(pr)))
sp = per_run.pivot(index="seed", columns="condition", values="spikes_in_condition")
print("spikes over the whole condition (all 100 repeats), hyperpolarizing / shunting: %.3f to %.3f"
      % ((sp.hyperpolarizing / sp.shunting).min(), (sp.hyperpolarizing / sp.shunting).max()))

**Result: reproduced.** The peri-stimulus rate is lower with hyperpolarizing E_GABA, 8.785 against 19.468 Hz over 16,000 neurons (paired t = 446.42, p below 0.05), and lower in all 20 runs; over whole conditions the hyperpolarizing network fires 0.431 to 0.445 times the spikes of the shunting one. The spike-time parts of the statement hold in the two claims above (higher entropy and lower synchrony with shunting E_GABA).

## figs7-model-decoding

> Responses of networks with shunting E_GABAAR to 100 different input patterns are decoded more
> accurately by a downstream classifier.

The gist's decoder: spike counts of the first 50 excitatory neurons in each 25 ms epoch, a
k-nearest neighbour classifier (scikit-learn defaults), 5-fold shuffled stratified
cross-validation over the 10,000 trials of a condition, mean accuracy. Chance is 1%. Readings: (D1)
paired t test of the 20 per-run accuracies, (D2) shunting higher in every run.

In [ ]:
acc = per_run.pivot(index="seed", columns="condition", values="accuracy")
t = stats.ttest_rel(acc.shunting, acc.hyperpolarizing)
print(acc.round(4).to_string())
print(f"\nmean accuracy shunting {acc.shunting.mean():.4f}, hyperpolarizing {acc.hyperpolarizing.mean():.4f} (chance 0.01)")
print(f"D1 paired t = {t.statistic:.2f}, p = {t.pvalue:.2e}")
print("D2 shunting higher in %d of %d runs" % ((acc.shunting > acc.hyperpolarizing).sum(), len(acc)))
print(f"\nnotebook run time {(time.time() - T0) / 60:.1f} min")

**Result: reproduced.** Decoding accuracy is higher with shunting E_GABA, 0.9848 against 0.9315 on average (chance 0.01); D1 paired t = 16.42, p = 1.11e-12, and D2 holds, shunting higher in 20 of 20 runs.

## Summary

| Claim | Outcome |
|---|---|
| fig4h-model-synchrony | partially reproduced (hyperpolarizing mean 0.8789, 0.0011 outside tolerance) |
| fig4h-model-entropy | partially reproduced (hyperpolarizing mean 0.2014, 0.0024 outside tolerance) |
| fig4f-model-hyperpolarizing-less-activity | reproduced |
| figs7-model-decoding | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

The two partial outcomes are means a little outside tolerances that are the SEM over pooled
neurons; the spread of the per-run means is wider (the standard error of a 20-run mean is 0.0013
for synchrony and 0.0023 for entropy, and the misses are 1.6 and 1.9 of these), and the paper's
seeds are not known.

**How the analysis differs from the paper's.** The model is the paper's code, with the stimulus held
as one periodic set to fit in memory and seeds fixed, so the 20 runs are other draws of the same
model than the paper's. On a small run with one seed, `network.py` gives the same spikes as its
own whole-run stimulus and as the gist itself run unmodified, and the same per-neuron rate,
entropy and synchrony as the gist; in every full run the learning rate in the compiled synapse
code is 0.1 during balancing and 0 during the test (checked above).